# Cascaded RC filter analysis

This notebook follows *Reading Data Using Python* steps 1 to 7 of the proposal. It reads the files written by `daq.py` (or SerialPlot recordings saved with the same names) from the folder set in `DATA`.

Before real data exist, run `python simulate_data.py` once and keep `DATA = Path("data_sim")`. The simulated circuits use slightly different "true" component values, so every comparison below should land within about two standard deviations.

**Expected file layout**

| Folder | Files |
|---|---|
| `calib/` | `stream_1500Hz.csv`, `burst_1500Hz.csv`, `delay_200Hz.csv`, `delay_1000Hz.csv`, `delay_3000Hz.csv`, `triangle_1Hz.csv`, optional `noise_dc.csv` |
| `sweeps/` | `ref*.csv`, `lp1*.csv`, `lp2*.csv`, `bpL*.csv`, `bpB*.csv` (one or more files each, every complete sweep inside is used) |
| `phase/` | `lp1_<f>Hz.csv`, `bpL_<f>Hz.csv`, `bpB_<f>Hz.csv` for the nine frequencies |
| `square/` | `bpL_1.csv` to `bpL_3.csv`, `bpB_1.csv` to `bpB_3.csv` |

`bpL` is the loaded band-pass (Circuit 3) and `bpB` the buffered one (Circuit 4).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rcfilter as rc

DATA = Path("data_sim")            # change to Path("data") for the real recordings
F1, F2, T_SWEEP = 20.0, 10000.0, 20.0
PHASE_F = 2000.0                   # frequency of the quoted phase lag
N_MC = 10000
BOUND_IS = "rectangular"           # or "one_sigma" (see rc.draw_parts)
PAIR_US, TRI_US = 49, 200          # burst periods used for the two-channel and triangle records
pd.set_option("display.float_format", lambda v: f"{v:.5g}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## Predictions from the measured components (step 7, run first)

Replace each value with that part's own reading, so that R1, R2 and R3 (and C1 to C4) keep their individual values, and set the accuracy terms from the multimeter manual. Each part reads as $\pm(\text{pct}\,\% \text{ of reading} + \text{counts} \times \text{resolution})$.

In [ ]:
RES_1K  = dict(pct=0.8, counts=2, resolution=1.0)      # 2 kΩ range, 1 Ω per count
RES_10K = dict(pct=0.8, counts=2, resolution=10.0)     # 20 kΩ range, 10 Ω per count
CAP     = dict(pct=3.0, counts=5, resolution=0.1e-9)   # 200 nF range, 0.1 nF per count

parts = {
    "R1": rc.part(987.0, **RES_1K),  "R2": rc.part(987.0, **RES_1K),  "R3": rc.part(987.0, **RES_1K),
    "R4": rc.part(9970.0, **RES_10K), "R5": rc.part(9970.0, **RES_10K),
    "C1": rc.part(99.7e-9, **CAP), "C2": rc.part(99.7e-9, **CAP),
    "C3": rc.part(99.7e-9, **CAP), "C4": rc.part(99.7e-9, **CAP),
    "Rg": rc.part(50.0, pct=2.0),                       # generator output resistance, nominal 50 Ω
}
draws = rc.draw_parts(parts, N_MC, seed=1, bound_is=BOUND_IS)
mc = rc.predict(draws, phase_f=PHASE_F)
nominal = rc.predict({k: v["value"] for k, v in parts.items()}, phase_f=PHASE_F)

def std_unc(name):
    b = parts[name]["bound"]
    return b / np.sqrt(3) if BOUND_IS == "rectangular" else b

show = [("lp1.fc_sweep", "single stage f_c, sweep", "Hz", 1), ("lp1.fc_phase", "single stage f_c, phase", "Hz", 1),
        ("lp2.b_loaded", "two stage b, loaded", "1e-4 s", 1e4), ("lp2.b_ideal", "two stage b, ideal", "1e-4 s", 1e4),
        ("lp2.fhalf_ideal", "two stage f_1/2, ideal", "Hz", 1), ("lp2.fhalf_loaded", "two stage f_1/2, loaded", "Hz", 1),
        ("bpB.b", "band-pass b, buffered", "1e-4 s", 1e4), ("bpL.b", "band-pass b, loaded", "1e-4 s", 1e4),
        ("bp.c_change_pct", "change in c from loading", "%", 1),
        ("bpB.f_lo", "edge low, buffered", "Hz", 1), ("bpB.f_hi", "edge high, buffered", "Hz", 1),
        ("bpL.f_lo", "edge low, loaded", "Hz", 1), ("bpL.f_hi", "edge high, loaded", "Hz", 1),
        ("bpB.df", "bandwidth, buffered", "Hz", 1), ("bpL.df", "bandwidth, loaded", "Hz", 1),
        ("bpB.Q", "Q, buffered", "", 1), ("bpL.Q", "Q, loaded", "", 1),
        ("bpB.f0", "f0, buffered", "Hz", 1), ("bpL.f0", "f0, loaded", "Hz", 1),
        ("bpB.peak_power", "peak power ratio, buffered", "", 1), ("bpL.peak_power", "peak power ratio, loaded", "", 1),
        ("bpB.product", "(K/b)·Δf, buffered", "Hz", 1), ("bpL.product", "(K/b)·Δf, loaded", "Hz", 1),
        ("bp.ratio_20Hz", "P_loaded / P_buffered at 20 Hz", "", 1),
        ("bpB.ph_f0", "phase 0° crossing, buffered", "Hz", 1), ("bpL.ph_f0", "phase 0° crossing, loaded", "Hz", 1),
        ("bpB.ph_p45", "+45° point, buffered", "Hz", 1), ("bpB.ph_m45", "−45° point, buffered", "Hz", 1),
        ("bpL.ph_p45", "+45° point, loaded", "Hz", 1), ("bpL.ph_m45", "−45° point, loaded", "Hz", 1),
        (f"bpB.ph_{int(PHASE_F)}Hz", f"phase at {PHASE_F:.0f} Hz, buffered", "deg", 1),
        (f"bpL.ph_{int(PHASE_F)}Hz", f"phase at {PHASE_F:.0f} Hz, loaded", "deg", 1)]
for tag, form in (("bpL", "loaded"), ("bpB", "buffered")):
    show += [(f"{tag}.norm{f}", f"|H_{f}|/|H_500|, {form}", "", 1) for f in (1500, 2500, 3500)]
    show += [(f"{tag}.H{f}", f"|H| at {f} Hz, {form}", "", 1) for f in (500, 1500, 2500, 3500)]
    show += [(f"{tag}.spread_deg", f"phase spread of 4 harmonics, {form}", "deg", 1)]
pred_table = pd.DataFrame([(lab, nominal[k] * s, np.mean(mc[k]) * s, np.std(mc[k], ddof=1) * s, u)
                           for k, lab, u, s in show],
                          columns=["prediction", "nominal parts", "MC mean", "MC std", "unit"]).set_index("prediction")
pred_table

## Step 1: ESP32 calibration

Sampling rates come from a sine fit to the 1500 Hz recordings. The delay between paired samples comes from fitting the phase difference of the two pins on the same signal to

$$\Delta\varphi = 2\pi f\,\Delta t .$$

The 1 Hz triangle wave from 0.9 V to 2.4 V checks that the ADC is linear over the signal range.

In [ ]:
fs_stream, s_fs_stream = rc.sampling_rate(rc.load_recording(DATA / "calib/stream_1500Hz.csv")[:, 0], 1500.0)
b15 = rc.load_recording(DATA / "calib/burst_1500Hz.csv")
fsA, sA = rc.sampling_rate(b15[:, 0], 1500.0)
fsB, sB = rc.sampling_rate(b15[:, 1], 1500.0)
fs_pair = (fsA / sA**2 + fsB / sB**2) / (1 / sA**2 + 1 / sB**2)
s_fs_pair = 1 / np.sqrt(1 / sA**2 + 1 / sB**2)
print(f"stream rate      {fs_stream:.3f} ± {s_fs_stream:.3f} S/s")
print(f"two-channel rate {fs_pair:.3f} ± {s_fs_pair:.3f} S/s per channel (GPIO 34 {fsA:.3f}, GPIO 35 {fsB:.3f})")

recs = []
for p in sorted((DATA / "calib").glob("delay_*Hz.csv"), key=rc.freq_from_name):
    d = rc.load_recording(p)
    recs.append((rc.freq_from_name(p), d[:, 0], d[:, 1]))
delay = rc.channel_delay(recs, fs_pair)
DT, S_DT, GAIN = delay["dt"], delay["sdt_scaled"], delay["gain_ratio"]
print(f"channel delay Δt = {DT*1e6:.3f} ± {S_DT*1e6:.3f} µs (reduced χ² {delay['chi2_red']:.2f})")
print(f"gain ratio GPIO35/GPIO34 = {GAIN:.4f} ± {delay['s_gain_ratio']:.4f}")
display(delay["table"])

tri_d = rc.load_recording(DATA / "calib/triangle_1Hz.csv")
fs_tri = fs_pair * PAIR_US / TRI_US
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for col, pin in ((0, 34), (1, 35)):
    tri = rc.triangle_linearity(tri_d[:, col], fs_tri, 0.9, 2.4)
    print(f"GPIO {pin}: {tri['counts_per_volt']:.1f} counts/V, deviation from a line "
          f"rms {tri['resid_rms_mV']:.2f} mV, max {tri['resid_max_mV']:.2f} mV "
          f"({tri['max_dev_pct_of_span']:.2f} % of the 1.5 V span)")
    m = tri["mask"]
    ax[col].plot(tri["volts"][m], tri["resid_mV"][m], ".", ms=1, alpha=0.4)
    ax[col].set(xlabel="input voltage (V)", ylabel="residual (mV)", title=f"Triangle linearity, GPIO {pin}")
plt.tight_layout(); plt.show()

## Step 2: frequency sweeps to power ratios

Each complete sweep is found automatically, trimmed, mean subtracted, squared, smoothed over one local period, and divided by the mean reference sweep. Time converts to frequency with

$$f(t) = f_1\left(\frac{f_2}{f_1}\right)^{t/T}.$$

Bins above $0.4 f_s$ are excluded, because the ESP32 has no anti-alias filter. With the 40 kS/s binary stream the full 10 kHz range is used.

In [ ]:
FIT_LO, FIT_HI = 1.25 * F1, min(F2 / 1.15, 0.4 * fs_stream)
noise_file = DATA / "calib/noise_dc.csv"
NOISE_VAR = rc.noise_variance(rc.load_recording(noise_file)[:, 0]) if noise_file.exists() else 0.0
print(f"fit range {FIT_LO:.0f} Hz to {FIT_HI:.0f} Hz, ADC noise variance {NOISE_VAR:.2f} counts²")

def get_sweeps(name):
    paths = sorted((DATA / "sweeps").glob(f"{name}*.csv"))
    segs, found = rc.sweeps_from_files(paths, fs_stream, F1, F2, T_SWEEP, noise_var=NOISE_VAR)
    for path, starts in found:
        print(f"  {Path(path).name}: sweeps start at {', '.join(f'{s:.3f}' for s in starts)} s")
    return segs

print("reference");  refs = get_sweeps("ref")
names = {"lp1": "single-stage low-pass", "lp2": "two-stage low-pass", "bpL": "band-pass loaded", "bpB": "band-pass buffered"}
SW = {}
for key, label in names.items():
    print(label)
    SW[key] = rc.sweep_ratio(get_sweeps(key), refs, FIT_LO, FIT_HI, n_bins=120)
    print(f"  {SW[key]['n_out']} sweeps against {SW[key]['n_ref']} reference sweeps")

x = rc.load_recording(sorted((DATA / "sweeps").glob("bpL*.csv"))[0])[:, 0]
starts, diag = rc.sweep_starts(x, fs_stream, F1, F2, T_SWEEP, return_diag=True)
fig, ax = plt.subplots(figsize=(10, 3))
ax.semilogy(diag["t"], diag["fpk"], ".", ms=2, color="0.6", label="spectrogram peak")
ax.semilogy(diag["t"][diag["good"]], diag["fpk"][diag["good"]], ".", ms=2, label="used for the start time")
for s in starts:
    ax.axvline(s, color="C3", lw=1)
ax.set(xlabel="time in recording (s)", ylabel="frequency (Hz)", title="Sweep start detection (red lines)")
ax.legend(loc="lower right"); plt.show()

## Step 3: low-pass fits

The single stage is fitted to $P/P_0 = 1/[1+(\omega\tau)^2]$. The two stage is fitted with $b$ and $c$ free to

$$\frac{P}{P_0} = \frac{1}{(1-c\omega^2)^2+(b\omega)^2}.$$

The loaded and ideal models share this form and the same $c$, so with $b$ and $c$ free they give identical fits. The test is therefore the fitted $b$ against each prediction, plus the reduced $\chi^2$ of each model evaluated at its predicted parameters.

In [ ]:
def scaled(fit):
    return np.sqrt(max(1.0, fit.chi2_red))

fit1 = rc.fit_lp1_sweep(SW["lp1"])
tau1, s_tau1 = fit1.p[0], fit1.perr_scaled[0]
fc1, s_fc1 = 1 / (2 * np.pi * tau1), s_tau1 / (2 * np.pi * tau1**2)
print(f"single stage: τ = {tau1*1e6:.3f} ± {s_tau1*1e6:.3f} µs, f_c = {fc1:.1f} ± {s_fc1:.1f} Hz, reduced χ² {fit1.chi2_red:.2f}")

fit2 = rc.fit_lp2_sweep(SW["lp2"])
print(fit2)
P_mean = {k: float(np.mean(v)) for k, v in mc.items()}
sw = SW["lp2"]
for label, bkey in (("loaded", "lp2.b_loaded"), ("ideal", "lp2.b_ideal")):
    c2 = rc.chi2_at(rc.P_lp2, sw["f"], sw["ratio"], sw["sigma"], (P_mean[bkey], P_mean["lp2.c"]))
    z = rc.zscore(fit2.p[0], fit2.perr_scaled[0], P_mean[bkey], np.std(mc[bkey], ddof=1))
    print(f"{label:7s} model at predicted b, c: reduced χ² = {c2['chi2_red']:.3g};  fitted b vs predicted: z = {z:+.2f}")
fh2, s_fh2 = rc.lp2_half_power_from_fit(fit2)
s_fh2 *= scaled(fit2)
print(f"two-stage half-power frequency {fh2:.1f} ± {s_fh2:.1f} Hz")

fig, axs = plt.subplots(2, 2, figsize=(11, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for j, (key, fn, p, ttl) in enumerate((("lp1", rc.P_lp1, fit1.p, "Single stage"), ("lp2", rc.P_lp2, fit2.p, "Two stage"))):
    s = SW[key]
    axs[0, j].errorbar(s["f"], s["ratio"], s["sigma"], fmt=".", ms=3, label="data")
    axs[0, j].plot(s["f"], fn(s["f"], *p), label="fit")
    if key == "lp2":
        axs[0, j].plot(s["f"], rc.P_lp2(s["f"], P_mean["lp2.b_ideal"], P_mean["lp2.c"]), "--", label="ideal prediction")
    axs[0, j].set(xscale="log", yscale="log", ylabel="P / P0", title=ttl); axs[0, j].legend()
    axs[1, j].plot(s["f"], (s["ratio"] - fn(s["f"], *p)) / s["sigma"], ".", ms=3)
    axs[1, j].set(xscale="log", xlabel="frequency (Hz)", ylabel="residual / σ")
plt.tight_layout(); plt.show()

## Step 4: band-pass fits and the model-free checks

Each form is fitted to

$$\frac{P}{P_0} = \frac{(K\omega)^2}{(1-c\omega^2)^2+(b\omega)^2},$$

from which $f_0 = 1/(2\pi\sqrt{c})$, $\Delta f = b/(2\pi c)$, $Q=\sqrt{c}/b$, the peak $K/b$ and the product $(K/b)\,\Delta f$ follow. The half-power edges are also read directly from the smoothed curves, and the loaded curve is divided by the buffered curve.

In [ ]:
BP = {}
for key in ("bpL", "bpB"):
    fit = rc.fit_bp_sweep(SW[key])
    der = rc.bp_derived_from_fit(fit)
    k = scaled(fit)
    der = {n: (v, e * k) for n, (v, e) in der.items()}
    BP[key] = {"fit": fit, "der": der, "free": rc.model_free_edges(SW[key])}
    print(names[key], fit)

rows = []
for n in ("f0", "df", "Q", "peak_amp", "peak_power", "product", "f_lo", "f_hi"):
    rows.append([n] + [x for key in ("bpL", "bpB") for x in BP[key]["der"][n]])
display(pd.DataFrame(rows, columns=["quantity", "loaded", "± loaded", "buffered", "± buffered"]).set_index("quantity"))
rows = [[n] + [x for key in ("bpL", "bpB") for x in BP[key]["free"][n]] for n in ("f_lo", "f_hi", "df", "peak_power")]
print("model-free (read directly from the smoothed curves):")
display(pd.DataFrame(rows, columns=["quantity", "loaded", "± loaded", "buffered", "± buffered"]).set_index("quantity"))

cL, cB = BP["bpL"]["fit"]["c"], BP["bpB"]["fit"]["c"]
c_change = 100 * (cL[0] / cB[0] - 1)
s_c_change = 100 * cL[0] / cB[0] * np.hypot(cL[1] / cL[0], cB[1] / cB[0]) * max(scaled(BP["bpL"]["fit"]), scaled(BP["bpB"]["fit"]))
print(f"change in c from loading: {c_change:.1f} ± {s_c_change:.1f} %")

LB = rc.loaded_over_buffered(SW["bpL"], SW["bpB"])
fl = LB["lowest_f"]
qpred = rc.P_bp(fl, mc["bpL.K"], mc["bpL.b"], mc["bpL.c"]) / rc.P_bp(fl, mc["bpB.K"], mc["bpB.b"], mc["bpB.c"])
print(f"loaded / buffered: maximum {LB['max_q']:.4f} ± {LB['max_q_sigma']:.4f} at {LB['f_at_max']:.0f} Hz; "
      f"bins above 1 by more than 2σ: {LB['n_above_1_by_2sigma']}")
print(f"at the lowest bin ({fl:.1f} Hz): {LB['q_lowest']:.4f} ± {LB['s_q_lowest']:.4f}, predicted {np.mean(qpred):.4f} ± {np.std(qpred):.4f}")

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for key, col in (("bpB", "C0"), ("bpL", "C1")):
    s = SW[key]
    axs[0].errorbar(s["f"], s["ratio"], s["sigma"], fmt=".", ms=3, color=col, label=names[key])
    axs[0].plot(s["f"], rc.P_bp(s["f"], *BP[key]["fit"].p), color=col)
axs[0].set(xscale="log", xlabel="frequency (Hz)", ylabel="P / P0", title="Band-pass sweeps and fits"); axs[0].legend()
axs[1].errorbar(LB["f"], LB["q"], LB["sigma"], fmt=".", ms=3)
axs[1].axhline(1, color="k", lw=0.8)
ff = np.geomspace(FIT_LO, FIT_HI, 300)
axs[1].plot(ff, rc.P_bp(ff, P_mean["bpL.K"], P_mean["bpL.b"], P_mean["bpL.c"]) / rc.P_bp(ff, P_mean["bpB.K"], P_mean["bpB.b"], P_mean["bpB.c"]), label="prediction")
axs[1].set(xscale="log", xlabel="frequency (Hz)", ylabel="P_loaded / P_buffered", title="Loaded over buffered"); axs[1].legend()
plt.tight_layout(); plt.show()

### Figure A: the phenomenon

The left panel normalizes each curve to its own peak, which is how the half-power bandwidth is defined, so the wider loaded band is visible directly from the data. The right panel keeps the absolute scale and marks one fixed level (the loaded filter's half-power level). At that level the buffered band is the wider one, because the loaded response lies below the buffered response at every frequency: the widening comes entirely from the lower peak.

In [ ]:
COL = {"bpB": "#2a78d6", "bpL": "#eb6834"}   # buffered blue, loaded orange (validated categorical pair)
LEVEL = BP["bpL"]["free"]["peak_power"][0] / 2
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
for key in ("bpB", "bpL"):
    s, free = SW[key], BP[key]["free"]
    lo, hi, df = free["f_lo"][0], free["f_hi"][0], free["df"][0]
    form = "loaded" if key == "bpL" else "buffered"
    ax[0].plot(s["f"], s["ratio"] / free["peak_power"][0], "o", ms=2.5, color=COL[key],
               label=f"{form}: Δf = {df:.0f} Hz")
    ax[0].axvspan(lo, hi, color=COL[key], alpha=0.10, lw=0)
    for e in (lo, hi):
        ax[0].axvline(e, color=COL[key], lw=1, ls="--")
    band = rc.band_at_level(s["f"], s["ratio"], LEVEL)
    ax[1].plot(s["f"], s["ratio"], "o", ms=2.5, color=COL[key],
               label=f"{form}: {band['f_lo']:.0f} to {band['f_hi']:.0f} Hz")
    yb = LEVEL + (0.03 if key == "bpB" else -0.03)   # offset so the two bands do not overlap
    ax[1].plot([band["f_lo"], band["f_hi"]], [yb, yb], color=COL[key], lw=2.5, marker="|", ms=12, mew=2)
    print(f"{names[key]:20s} half-power band {lo:7.1f} to {hi:7.1f} Hz (Δf = {df:6.1f} Hz); "
          f"band above P/P0 = {LEVEL:.3f}: {band['f_lo']:7.1f} to {band['f_hi']:7.1f} Hz ({band['width']:6.1f} Hz)")
ax[0].axhline(0.5, color="0.35", lw=0.8)
ax[0].set(xscale="log", ylim=(0, 1.08), xlabel="frequency (Hz)", ylabel="P / P_peak (each curve's own peak)",
          title="Half-power bandwidth: loading widens the band")
ax[1].axhline(LEVEL, color="0.35", lw=0.8)
ax[1].set(xscale="log", xlabel="frequency (Hz)", ylabel="P / P0",
          title=f"Absolute scale: band above P/P0 = {LEVEL:.3f}")
for a in ax:
    a.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

## Step 5: phase points

Both channels are fitted to sines at the input frequency and the delay is removed, $\varphi = \arg(Z_\text{out}/Z_\text{in}) - 2\pi f\,\Delta t$. The single stage gives $\tau$ without $R_g$, so the two single-stage cutoffs give the generator resistance

$$R_g = R_1\left(\frac{\tau_\text{sweep}}{\tau_\text{phase}} - 1\right).$$

The band-pass phase is $\varphi = 90^\circ - \arg(1 - c\omega^2 + jb\omega)$.

In [ ]:
PH = {key: rc.phase_set(sorted((DATA / "phase").glob(f"{key}_*Hz.csv")), fs_pair, DT, S_DT, GAIN)
      for key in ("lp1", "bpL", "bpB")}
fitp1 = rc.fit_lp1_phase(PH["lp1"])
taup, s_taup = fitp1.p[0], fitp1.perr_scaled[0]
fcp, s_fcp = 1 / (2 * np.pi * taup), s_taup / (2 * np.pi * taup**2)
print(f"single stage phase: f_c = {fcp:.1f} ± {s_fcp:.1f} Hz, reduced χ² {fitp1.chi2_red:.2f}")
Rg_m, s_Rg_m = rc.generator_resistance(tau1, s_tau1, taup, s_taup, parts["R1"]["value"], std_unc("R1"))
print(f"generator resistance from the two cutoffs: Rg = {Rg_m:.1f} ± {s_Rg_m:.1f} Ω")

PM = {}
for key in ("bpL", "bpB"):
    fit = rc.fit_bp_phase(PH[key])
    marks = rc.bp_phase_marks(fit, PHASE_F)
    PM[key] = {"fit": fit, "marks": {n: (v, e * scaled(fit)) for n, (v, e) in marks.items()}}
    print(names[key], fit)
display(pd.DataFrame({names[k]: {n: f"{v:.4g} ± {e:.2g}" for n, (v, e) in PM[k]["marks"].items()} for k in PM}))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ff = np.geomspace(50, 8000, 300)
ax[0].errorbar(PH["lp1"].f, PH["lp1"].phase_deg, PH["lp1"].s_phase_deg, fmt="o", ms=4)
ax[0].plot(ff, rc.phase_lp1_deg(ff, taup))
ax[0].set(xscale="log", xlabel="frequency (Hz)", ylabel="phase (deg)", title="Single stage, output vs terminal")
for key, col in (("bpB", "C0"), ("bpL", "C1")):
    ax[1].errorbar(PH[key].f, PH[key].phase_deg, PH[key].s_phase_deg, fmt="o", ms=4, color=col, label=names[key])
    ax[1].plot(ff, rc.phase_bp_deg(ff, *PM[key]["fit"].p), color=col)
for y in (45, 0, -45):
    ax[1].axhline(y, color="k", lw=0.5)
ax[1].set(xscale="log", xlabel="frequency (Hz)", ylabel="phase (deg)", title="Band-pass phase"); ax[1].legend()
plt.tight_layout(); plt.show()

## Step 6: square-wave bursts

Each burst is trimmed to whole periods of the fitted fundamental, and every harmonic below the Nyquist frequency is measured at exactly $k f_0$, which equals the DFT of the trimmed record. The output predicted from the measured input harmonics and each model (terminal view, no $R_g$) is compared with the recording.

In [ ]:
def load_bursts(key):
    out = []
    for p in sorted((DATA / "square").glob(f"{key}_*.csv")):
        d = rc.load_recording(p)
        out.append(rc.square_burst(d[:, 0], d[:, 1], fs_pair, 500.0, DT, GAIN))
    return out

SQ = {key: load_bursts(key) for key in ("bpL", "bpB")}
SQS = {key: rc.square_summary(SQ[key]) for key in SQ}
display(pd.concat({names[k]: SQS[k] for k in SQS}, axis=1))

SQP = {}
for key in SQ:
    Kt, bt, ct = (P_mean[f"{key}.{n}"] for n in ("Kt", "bt", "ct"))
    res = [rc.predict_square_output(bu, Kt, bt, ct) for bu in SQ[key]]
    SQP[key] = res
    fr = [r[2] for r in res]
    print(f"{names[key]}: RMS residual of the predicted output = {100*np.mean(fr):.2f} % of peak-to-peak "
          f"(each burst: {', '.join(f'{100*v:.2f}' for v in fr)} %; criterion 3 %)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for j, key in enumerate(("bpB", "bpL")):
    bu, (y, rms, fr) = SQ[key][0], SQP[key][0]
    m = bu["t"] < 4e-3
    ax[j].plot(bu["t"][m] * 1e3, bu["xA"][m], ".", ms=2, color="0.6", label="input (GPIO 34)")
    ax[j].plot(bu["t"][m] * 1e3, bu["xB"][m], ".", ms=3, label="output (GPIO 35)")
    ax[j].plot(bu["t"][m] * 1e3, y[m], lw=1.2, label="predicted output")
    ax[j].set(xlabel="time (ms)", ylabel="ADC counts", title=names[key]); ax[j].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Figure B: the advantage of each form

Bars are the measured square-wave harmonics (mean of the bursts, with standard errors), and black ticks are the predictions. The left panel normalizes each harmonic to the fundamental: the loaded filter passes its harmonics more evenly, so it changes the shape of a broadband signal less. The right panel shows absolute transmission: the buffered filter passes more of every harmonic.

In [ ]:
from matplotlib.lines import Line2D
COL = {"bpB": "#2a78d6", "bpL": "#eb6834"}
HF = [500, 1500, 2500, 3500]
x, w = np.arange(len(HF)), 0.38
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
for j, key in enumerate(("bpB", "bpL")):
    S, xs = SQS[key], x + (j - 0.5) * w
    norm = [1.0] + [S.loc[f"norm{f}", "mean"] for f in HF[1:]]
    s_norm = [0.0] + [S.loc[f"norm{f}", "sem"] for f in HF[1:]]
    p_norm = [1.0] + [np.mean(mc[f"{key}.norm{f}"]) for f in HF[1:]]
    spread = S.loc["spread_deg", "mean"]
    ax[0].bar(xs, norm, w * 0.92, yerr=s_norm, color=COL[key], capsize=2,
              label=f"{names[key]} (phase spread {spread:.0f}°)")
    ax[0].plot(xs, p_norm, "_", color="k", ms=16, mew=2)
    absH = [S.loc[f"H{f}", "mean"] for f in HF]
    s_abs = [S.loc[f"H{f}", "sem"] for f in HF]
    ax[1].bar(xs, absH, w * 0.92, yerr=s_abs, color=COL[key], capsize=2, label=names[key])
    ax[1].plot(xs, [np.mean(mc[f"{key}.H{f}"]) for f in HF], "_", color="k", ms=16, mew=2)
    for xi, v in zip(xs[1:], norm[1:]):
        ax[0].text(xi, v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
pred_handle = Line2D([], [], color="k", marker="_", ls="", ms=16, mew=2, label="prediction")
labels = [f"{f} Hz\n(harmonic {k})" for f, k in zip(HF, (1, 3, 5, 7))]
ax[0].set(xticks=x, xticklabels=labels, ylim=(0, 1.15), ylabel="|H_k| / |H_1|",
          title="Loaded: harmonics passed more evenly")
ax[1].set(xticks=x, xticklabels=labels, ylim=(0, 1.0), ylabel="|H_k|",
          title="Buffered: more signal at every harmonic")
for a in ax:
    h, l = a.get_legend_handles_labels()
    a.legend(h + [pred_handle], l + ["prediction"], fontsize=8, loc="upper right")
    a.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

## Step 7: every prediction against the data

$$z = \frac{x_\text{measured} - x_\text{predicted}}{\sqrt{\sigma_\text{measured}^2 + \sigma_\text{predicted}^2}}$$

Measured errors are inflated by $\sqrt{\chi^2_\text{red}}$ whenever a fit's reduced $\chi^2$ exceeds 1. The row "two stage b against ideal" is expected to fail, because that is the loading effect.

In [ ]:
R = rc.Report(mc)
R.add("single stage f_c, sweep", fc1, s_fc1, key="lp1.fc_sweep", unit="Hz")
R.add("single stage f_c, phase", fcp, s_fcp, key="lp1.fc_phase", unit="Hz")
R.add("generator resistance Rg", Rg_m, s_Rg_m, parts["Rg"]["value"], std_unc("Rg"), unit="Ω")
R.add("two stage b against loaded", fit2.p[0], fit2.perr_scaled[0], key="lp2.b_loaded", unit="1e-4 s", scale=1e4)
R.add("two stage b against ideal", fit2.p[0], fit2.perr_scaled[0], key="lp2.b_ideal", unit="1e-4 s", scale=1e4)
R.add("two stage c", fit2.p[1], fit2.perr_scaled[1], key="lp2.c", unit="1e-8 s²", scale=1e8)
R.add("two stage half-power frequency", fh2, s_fh2, key="lp2.fhalf_loaded", unit="Hz")
for key, form in (("bpL", "loaded"), ("bpB", "buffered")):
    fit, der, free = BP[key]["fit"], BP[key]["der"], BP[key]["free"]
    R.add(f"{form} b", fit.p[1], fit.perr_scaled[1], key=f"{key}.b", unit="1e-4 s", scale=1e4)
    R.add(f"{form} c", fit.p[2], fit.perr_scaled[2], key=f"{key}.c", unit="1e-8 s²", scale=1e8)
    for n, lab, u in (("f0", "center frequency", "Hz"), ("df", "bandwidth", "Hz"), ("Q", "Q", ""),
                      ("peak_power", "peak power ratio", ""), ("product", "(K/b)·Δf", "Hz"),
                      ("f_lo", "low edge", "Hz"), ("f_hi", "high edge", "Hz")):
        R.add(f"{form} {lab}", *der[n], key=f"{key}.{n}", unit=u)
    for n, lab in (("f_lo", "low edge"), ("f_hi", "high edge"), ("df", "bandwidth")):
        R.add(f"{form} {lab}, model-free", *free[n], key=f"{key}.{n}", unit="Hz")
    marks = PM[key]["marks"]
    R.add(f"{form} phase 0° crossing", *marks["f_zero"], key=f"{key}.ph_f0", unit="Hz")
    R.add(f"{form} +45° point", *marks["f_p45"], key=f"{key}.ph_p45", unit="Hz")
    R.add(f"{form} −45° point", *marks["f_m45"], key=f"{key}.ph_m45", unit="Hz")
    R.add(f"{form} phase at {PHASE_F:.0f} Hz", *marks[f"phase_{int(PHASE_F)}Hz"], key=f"{key}.ph_{int(PHASE_F)}Hz", unit="deg")
    S = SQS[key]
    for f in (500, 1500, 2500, 3500):
        R.add(f"{form} |H| at {f} Hz", S.loc[f"H{f}", "mean"], S.loc[f"H{f}", "sem"], key=f"{key}.H{f}")
    for f in (1500, 2500, 3500):
        R.add(f"{form} |H_{f}|/|H_500|", S.loc[f"norm{f}", "mean"], S.loc[f"norm{f}", "sem"], key=f"{key}.norm{f}")
    R.add(f"{form} phase spread of 4 harmonics", S.loc["spread_deg", "mean"], S.loc["spread_deg", "sem"], key=f"{key}.spread_deg", unit="deg")
R.add("change in c from loading", c_change, s_c_change, key="bp.c_change_pct", unit="%")
R.add(f"P_loaded/P_buffered at {fl:.0f} Hz", LB["q_lowest"], LB["s_q_lowest"], float(np.mean(qpred)), float(np.std(qpred)))
table = R.table()
print(f"{table.within_2sigma.sum()} of {len(table)} comparisons agree within 2σ")
table